# Step 3.3: the feature cache

This notebook is the check of step 3.3. `ff.features.compute` keeps each feature's values under `FEFAT_HOME/cache/features/`, split as the store is: a month of 1-minute bars, or a trade date of finer data. A second computation of the same features over the same dates reads them back instead of computing them again, while the data and the code they come from are unchanged; a backtest reads them there too.

Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR` and writes the feature cache in `FEFAT_HOME`, which `fefat cache features` lists and `fefat cache features --prune` trims; the cache can be deleted at any time and is computed again.

## A year computed twice

The example RSI and ATR, of 14 bars each, over the year of 5-minute bars to the last trade date of the product's data: MES when your store holds its 1-minute bars, else the product with the most contracts.

Expected: the product, the dates, and the rows; then the two calls' times: the first time you run this notebook, the first call computes the year, a few seconds, and the second reads the cache, a fraction of a second; when the cache already holds the year, both read it. Last, `the second read the same values: True`.

In [ ]:
import time
from datetime import timedelta

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.features.examples import atr, rsi

listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
FIRST = LAST - timedelta(days=365)
SLOTS = {"rsi": rsi(n=14), "atr": atr(n=14)}


def timed_compute() -> tuple[pl.DataFrame, float]:
    began = time.perf_counter()
    frame = ff.features.compute(SLOTS, ROOT, "5m", FIRST, LAST).collect()
    return frame, time.perf_counter() - began


first, computing = timed_compute()
again, reading = timed_compute()
print(f"{ROOT} from {FIRST} to {LAST}: {first.height:,} rows of 5-minute bars")
print(f"first call {computing:.2f} s, second call {reading:.2f} s")
print("the second read the same values:", again.equals(first))

## What the cache holds

Expected: a row for the product's 5-minute signal bars and one for each feature with its parameters, each with its partitions, the dates they span, their size in MB, and when one was last used; then the count of folders and partitions, which `fefat cache features` prints too.

In [ ]:
held = ff.features.cache_status().filter(pl.col("instrument") == ROOT, pl.col("timeframe") == "5m")
size_mb = (pl.col("size_bytes") / 1e6).round(1).alias("size_mb")
display(held.select("feature", "parameters", "partitions", "first", "last", size_mb, "last_used"))
print(f"{ROOT} 5m: {held.height} folders, {held.get_column('partitions').sum()} partitions")